# Embedding
  
Das Notebook dient dazu per:
  
- **CLIP**          CLIP -> 
- **AnyLoc**        DINOv2 -> Feature Aggregation -> Descriptor -> Retrival (Github: https://github.com/AnyLoc/Revisit-Anything.git)
- **EigenPlaces**   Backbone -> VPR-Descriptor -> Retrival (Github: https://github.com/gmberton/EigenPlaces.git)
- **MixVPR**        noch keine Idee (Mixed ansatz)
  
die Bilder in Vectorinformationen zu embedden


# Setup


In [ ]:

import pandas as pd
import yaml
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor
from pathlib import Path


def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")

def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None



PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
metadata = pd.read_parquet(DATA_PATH_META)


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


cfg_file = find_upwards("config.yaml")
assert cfg_file, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(cfg_file.read_text())
DATA_ROOT = CFG["dataset_folder"]
VPR = CFG["vpr"]["method"]
MODEL_ID = CFG["model"]
MODEL_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"   # generic after
IMAGE_DIR = DATA_ROOT / "raw" / "images"




# A GPU makes the image encoder roughly 20x faster, but nothing here *needs* one.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"running on: {DEVICE}")

# Plotting high resolution
plt.rcParams["figure.dpi"] = 300


# Modell laden


In [ ]:

model = CLIPModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)


print(f"Model:              {MODEL_ID}")
print(f"Device:             {DEVICE}")
print(f"parameters:         {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print(f"embedding size:     {model.config.projection_dim}")


with torch.no_grad():
    pixels = processor(images=batch, return_tensors="pt").to(DEVICE)
    emb = model.get_image_features(**pixels)
    emb = torch.nn.functional.normalize(emb, p = 2, dim=-1).cpu().numpy()

In [ ]:
# Bild laden

PROPS = {
    "id": "image_id",
    "sequence_id": "sequence_id",
    "captured_at": "captured_at",
    "compass_angle": "compass_angle",
    "is_pano": "is_pano",
    "creator_id": "creator_id",
    "split" : "split"
}

print(DATA_ROOT)
print(Path(DATA_ROOT).resolve())
data_root = Path(DATA_ROOT).resolve()

print(data_root.exists())


# Preprocessing

# Embedding Creation

In [ ]:
embedding_metadata = metadata[
    metadata["split"].isin(["database", "query"])
].copy()


def embed_images(image_paths, batch_size=32):
    """L2-normalised image embeddings, shape `(len(images), 512)`, as a numpy array.

    Accepts a single image, a single `(image, label)` pair, or a list of either.
    """
    emb = []
    for start in tqdm(range(0,len(image_pahts), batch_size), desc = "Bilder laden"):
        batch_paths = image_paths[start:start + batch_size]
        
        images = [
            Image.open(path).convert("RGB")
            for path in batch_paths
        ]
        
        with torch.no_grad():
            pixels = processor(images=images, return_tensors="pt").to(DEVICE)
            emb = model.get_image_features(**pixels).pooler_output
            emb = torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()

    return np.concatenate(emb)


image_paths = embedding_metadata["image_path"].tolist()

embeddings = embed_images(
    image_paths,
    batch_size = BATCH_SIZE
)

print("Shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

# Embedding Speichern